# Q6 - Real VLM responses and the frozen test protocol
These are genuine local SmolVLM generations, not simulated model outputs. This notebook only inspects completed evidence.

In [1]:
from pathlib import Path
import sys, json
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks': PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / 'src'))
from fabric_inspector.common import read_json
from IPython.display import display, Image as DisplayImage, Markdown
import pandas as pd
def read(path): return read_json(PROJECT / path)
def figure(path): display(DisplayImage(filename=str(PROJECT / path)))


In [2]:
from fabric_inspector.genai.vlm import parse_decision
selection=read('artifacts/vlm-selection.json')
display(selection)
for index in [0,1]:
    run=read(f'experiments/vlm-validation-{index}.json')
    assert len(run['examples'])==3472
    display({'prompt':run['config']['prompt'],'metrics':run['metrics'],'failure_rate':run['failure_rate']})
    display(run['examples'][:2])

{'status': 'COMPLETE',
 'split_sha256': 'af7daf13d0854ceeba004f1a06182bac778ac716021dec3d8fc40edf265aa76f',
 'provenance': 'RAW-FABRID_V1',
 'config': {'model_id': 'HuggingFaceTB/SmolVLM-256M-Instruct',
  'revision': '7e3e67edbbed1bf9888184d9df282b700a323964',
  'prompt': 'Classify this fabric patch. Look for a local disruption, hole, stain or anomalous yarn rather than ordinary repeating weave or lighting. Reply with only one label: defective or defect_free.',
  'prompt_index': 1,
  'max_new_tokens': 24,
  'do_sample': False,
  'quantisation': 'none',
  'dtype': 'bfloat16',
  'device': 'cpu',
  'cpu_threads': 1,
  'do_image_splitting': False,
  'processor_longest_edge': 256,
  'image_seq_len': 16,
  'response_format': 'label',
  'parsing_policy': 'Exact defective/defect_free token or one canonical sentence: The/This fabric (patch) is defective/defect-free. Any extra reasoning or ambiguity is unresolved.',
  'max_retries': 1,
  'failure_policy': 'Unresolved (-1) counts as an error for 

{'prompt': 'Inspect this fabric patch for a visible manufacturing defect. Reply with only one label: defective or defect_free.',
 'metrics': {'n': 3472,
  'tp': 52,
  'tn': 0,
  'fp': 3420,
  'fn': 0,
  'precision': 0.014976958525345621,
  'recall': 1.0,
  'f1': 0.029511918274687854,
  'f2': 0.07065217391304347,
  'accuracy': 0.014976958525345621,
  'false_positive_rate': 1.0,
  'unresolved_count': 74},
 'failure_rate': 0.02131336405529954}

[{'patch_id': 'Rollo6A_0000002_D_08-07-2024_11-18-23.246_0_0.png',
  'image_sha256': '2d2518fc65eff49018abc6f6a3db3011d8b5c660511469db51898a6a713145ab',
  'prediction': 1,
  'attempts': [{'raw_response': ' defective.',
    'latency_ms': 920.1873779998095,
    'input_tokens': 51,
    'output_tokens': 3,
    'resolved_revision': '7e3e67edbbed1bf9888184d9df282b700a323964'}],
  'end_to_end_ms': 922.5048369999058},
 {'patch_id': 'Rollo6A_0000002_D_08-07-2024_11-18-23.246_0_1.png',
  'image_sha256': '72ca1b413a00f99f603c64bbb1493a24d34960defcda1232814efe82ec35cb08',
  'prediction': 1,
  'attempts': [{'raw_response': ' defective.',
    'latency_ms': 955.6587039996884,
    'input_tokens': 51,
    'output_tokens': 3,
    'resolved_revision': '7e3e67edbbed1bf9888184d9df282b700a323964'}],
  'end_to_end_ms': 957.5681390001591}]

{'prompt': 'Classify this fabric patch. Look for a local disruption, hole, stain or anomalous yarn rather than ordinary repeating weave or lighting. Reply with only one label: defective or defect_free.',
 'metrics': {'n': 3472,
  'tp': 52,
  'tn': 0,
  'fp': 3420,
  'fn': 0,
  'precision': 0.014976958525345621,
  'recall': 1.0,
  'f1': 0.029511918274687854,
  'f2': 0.07065217391304347,
  'accuracy': 0.014976958525345621,
  'false_positive_rate': 1.0,
  'unresolved_count': 23},
 'failure_rate': 0.0066244239631336405}

[{'patch_id': 'Rollo6A_0000002_D_08-07-2024_11-18-23.246_0_0.png',
  'image_sha256': '2d2518fc65eff49018abc6f6a3db3011d8b5c660511469db51898a6a713145ab',
  'prediction': 1,
  'attempts': [{'raw_response': ' Defective.',
    'latency_ms': 593.4649769997122,
    'input_tokens': 66,
    'output_tokens': 4,
    'resolved_revision': '7e3e67edbbed1bf9888184d9df282b700a323964'}],
  'end_to_end_ms': 594.9173469998641},
 {'patch_id': 'Rollo6A_0000002_D_08-07-2024_11-18-23.246_0_1.png',
  'image_sha256': '72ca1b413a00f99f603c64bbb1493a24d34960defcda1232814efe82ec35cb08',
  'prediction': 1,
  'attempts': [{'raw_response': ' The fabric patch is defective.',
    'latency_ms': 757.9225700001189,
    'input_tokens': 66,
    'output_tokens': 7,
    'resolved_revision': '7e3e67edbbed1bf9888184d9df282b700a323964'}],
  'end_to_end_ms': 759.2079210003249}]

In [3]:
frozen=read('artifacts/final-evaluation-manifest.json')
guard=read('reports/final-evaluation/test-access.json')
vlm=read('reports/final-evaluation/vlm.json')
assert guard['status']=='COMPLETE'
assert [e['patch_id'] for e in vlm['examples']]==frozen['test_patch_ids']
assert len(vlm['examples'])==2996
assert vlm['labels_used_only_for_scoring']
display(vlm['metrics'])
print('All models share 2996 frozen test patches; unresolved generations remain in denominators.')

All models share 2996 frozen test patches; unresolved generations remain in denominators.


{'n': 2996,
 'tp': 94,
 'tn': 0,
 'fp': 2900,
 'fn': 2,
 'precision': 0.03139612558450234,
 'recall': 0.9791666666666666,
 'f1': 0.060841423948220064,
 'f2': 0.13913558318531677,
 'accuracy': 0.03137516688918558,
 'false_positive_rate': 1.0,
 'unresolved_count': 48}

In [4]:
display(read('reports/final-evaluation/comparison.json')['test_seed_repetitions'])
display(read('reports/application-verification/results.json')['status'])
display(Markdown((PROJECT/'docs/ai-use-declaration.md').read_text()))

{'f2': [0.6882591093117408, 0.7474747474747475, 0.6681514476614699],
 'mean': 0.7012951014826528,
 'sample_std': 0.041237110619281245}

'PASS'

# Declaration of AI assistance

OpenAI ChatGPT/Codex assisted with problem/protocol clarification, code repair and implementation, dataset acquisition and audit, experiment execution, tests, numerical analysis, notebook construction, report drafting and demonstration automation. The supplied project ZIP and lecturer notebooks informed the implementation. This declaration must accompany the individual submission; the student must understand and verify the generated material.

The assistant executed real software against the publisher's complete RAW-FABRID release and retained actual checkpoints, configurations, histories, predictions, explanations, timings and raw VLM responses. Report claims are based on those saved results. Earlier synthetic development checks are separately labelled and excluded from assessed metrics. Engineering interruptions and the workspace reset are retained separately from scientific unsuccessful approaches.

The assessed generative model in Q6 is a genuine local SmolVLM-256M-Instruct at a pinned publisher revision. Its image-only inference, prompts, output parsing and failures are recorded separately from the ChatGPT/Codex productivity assistance. No simulated VLM results satisfy Q6.

A runtime reset during a user pause lost the first completed final-evaluation files and recording. The same frozen protocol was physically recomputed to reconstruct them, with no retraining or altered model, threshold, prompt or split. Delivered outputs and timings are real reconstruction measurements; previously observed aggregate counts are cross-checks only. The recovery record discloses two physical computations of one benchmark, not independent repeated validation. The demonstration is an actual new recording.

The application video is a screen recording of scripted browser interaction with the actual local application, including fresh request-time inference, real Grad-CAM and a genuine incorrect prediction. It does not represent a student-led oral presentation. The student's personal review, authorship declaration under institutional rules, understanding of all code/results and individual live defence remain the student's responsibility and cannot be verified by an automated agent. No completed student presentation or approval letter is asserted; lecturer approval is recorded from the student's confirmation of the approved proposal.


No foundation-model class probability or class-only AUC is invented. Record the immutable revision, exact prompts, processor settings, token counts, raw outputs and local compute costs. A runtime reset lost the first completed final files; the identical frozen protocol was physically recomputed to reconstruct them, with no retraining or changed model/threshold/prompt/split. See docs/engineering-runs/final-results-workspace-reset.json. This is one benchmark, not independent repeated validation. Review the final report and viva guide: individual understanding and the live presentation must be demonstrated by the student.